In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/mahmoudshaheen1134/oil-sales-dataset/oil_sales_assignment_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer,make_column_transformer

In [ ]:
le=LabelEncoder()

In [ ]:
trf1=make_column_transformer((OneHotEncoder(sparse_output=False),['city','store_name','manufacturer','brand','class','price_bracket']),remainder='passthrough')

In [ ]:
df['size']=df['size'].str.replace("L","").astype(float)

In [ ]:
X=df.drop(columns=['sku','average_price'],axis=1)
y=df['average_price']

In [ ]:
df_new=pd.DataFrame(trf1.fit_transform(X))

In [ ]:
df_new

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(df_new,y,test_size=0.2,random_state=2)

In [ ]:
x_train

In [ ]:
from sklearn.linear_model import SGDRegressor

In [ ]:
sgd=SGDRegressor(penalty='l2',learning_rate='optimal',eta0=0.01)
sgd.fit(x_train,y_train)
r2_score(y_test,sgd.predict(x_test))

In [ ]:
from sklearn.svm import SVR

In [ ]:
svr=SVR()
svr.fit(x_train,y_train)
r2_score(y_test,svr.predict(x_test))

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

In [ ]:
dt=DecisionTreeRegressor()
np.mean(cross_val_score(dt,df_new,y,cv=10,scoring='r2'))

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor,AdaBoostRegressor,RandomForestRegressor,BaggingRegressor

In [ ]:
gbr=GradientBoostingRegressor()
abr=AdaBoostRegressor()
rf=RandomForestRegressor()

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
params_rf={'n_estimators':[10,50,100,200],"max_features":[0.25,0.50,1.0],"max_samples":[0.25,0.50,1.0]}
cv=GridSearchCV(rf,cv=5,scoring='r2',param_grid=params_rf,verbose=3)

In [ ]:
cv.fit(df_new,y)

In [ ]:
cv.best_score_

In [ ]:
params_abr={'n_estimators':[10,50,100,200,500]}
cv=GridSearchCV(abr,cv=5,scoring='r2',param_grid=params_abr,verbose=3)
cv.fit(df_new,y)

In [ ]:
cv.best_score_

In [ ]:
np.mean(cross_val_score(gbr,df_new,y,cv=5,scoring='r2'))

In [ ]:
params_gbr={'n_estimators':[10,50,100],'max_depth':[8,32,None]}
cv=GridSearchCV(gbr,cv=5,scoring='r2',param_grid=params_gbr,verbose=3)
cv.fit(df_new,y)

In [ ]:
cv.best_score_

In [ ]:
import xgboost as xgb

In [ ]:
model=xgb.XGBRegressor()
params_xgb={'n_estimators':[10,50,100],'max_depth':[8,32],'learning_rate':[0.01,0.1,1.0]}
cv=GridSearchCV(model,param_grid=params_xgb,cv=5,verbose=3,scoring='r2')
cv.fit(df_new,y)

In [ ]:
cv.best_score_

In [ ]:
from sklearn.ensemble import VotingRegressor

In [ ]:
estimators=[('dt',DecisionTreeRegressor()),('rf',RandomForestRegressor(max_features=0.5,max_samples=1.0,n_estimators=200)),('gb',GradientBoostingRegressor(max_depth=8)),('model',xgb.XGBRegressor(learning_rate=0.1,max_depth=8,n_estimators=100))]
np.mean(cross_val_score(VotingRegressor(estimators=estimators),df_new,y,cv=5,scoring='r2'))

# **Therefore, XGB has the highest cross_val  r2_score of 0.9919** 